In [1]:
# 06_robustness.ipynb
# Reviewer-requested robustness battery: (a) scaling of AUC loss and migration
# in Delta; (b) sensitivity to the number of features P; (c) banding scheme
# (equal-frequency / equal-width / cutoffs-at-modes); (d) failure of the
# equidistribution hypothesis under lattice-commensurate contributions.

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.metrics import roc_auc_score
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 110, "axes.grid": True,
                     "grid.color": "0.9", "font.size": 9, "axes.edgecolor": "0.3"})
GREYS = ["0.0", "0.3", "0.5", "0.65"]; LS = ["-", "--", "-.", ":"]; MK = ["o", "s", "^", "D"]
Path("../results/figures").mkdir(parents=True, exist_ok=True)
Path("../results/tables").mkdir(parents=True, exist_ok=True)
def _save(fig, name):
    fig.savefig(f"../results/figures/{name}.png", bbox_inches="tight")
    fig.savefig(f"../results/figures/{name}.pdf", bbox_inches="tight")

df = pd.read_parquet("../data/synthetic_population.parquet")
ccols = [c for c in df.columns if c.startswith("c")]
C = df[ccols].to_numpy(); f = df["f"].to_numpy(); p = df["p"].to_numpy(); y = df["y"].to_numpy()
P = len(ccols); b0 = float((f - C.sum(1)).mean())
def quant(A, D): return D * np.round(A / D)
def fq(D): return b0 + quant(C, D).sum(1)

def gen(Pv, N=20000, B=6, base=0.08, seed=0):
    r = np.random.default_rng(seed)
    base_strength = np.array([1.3, 1.1, 0.9, 0.8])
    strength = np.concatenate([base_strength, r.uniform(0.2, 0.6, max(Pv - 4, 0))])[:Pv]
    sig = r.normal(size=(N, 1)); z = 0.6 * sig + 0.8 * r.normal(size=(N, Pv))
    edges = np.quantile(z, np.linspace(0, 1, B + 1), axis=0)
    bi = np.empty((N, Pv), dtype=int)
    for j in range(Pv): bi[:, j] = np.clip(np.digitize(z[:, j], edges[1:-1, j]), 0, B - 1)
    bv = (np.arange(B) - (B - 1) / 2) / ((B - 1) / 2)
    Cm = np.stack([strength[j] * 0.45 * bv[bi[:, j]] for j in range(Pv)], axis=1)
    raw = Cm.sum(1); lo, hi = -20.0, 20.0
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        if (1 / (1 + np.exp(-(raw + mid)))).mean() > base: hi = mid
        else: lo = mid
    bb = mid; fc = raw + bb
    yy = (r.random(N) < 1 / (1 + np.exp(-fc))).astype(int)
    return Cm, fc, yy, bb

def migration(fc, fqv, K, scheme="freq"):
    if scheme == "freq":
        q = np.quantile(fc, np.linspace(0, 1, K + 1)[1:-1])
    elif scheme == "width":
        q = np.linspace(fc.min(), fc.max(), K + 1)[1:-1]
    else:
        hist, edg = np.histogram(fc, bins=200); centers = 0.5 * (edg[:-1] + edg[1:])
        q = np.sort(centers[np.argsort(hist)[-(K - 1):]])
    return float(np.mean(np.digitize(fc, q) != np.digitize(fqv, q)))

def reversal(fc, fqv, yy, npair=150000, seed=0):
    r = np.random.default_rng(seed); pos = np.where(yy == 1)[0]; neg = np.where(yy == 0)[0]
    i = r.choice(pos, npair); j = r.choice(neg, npair)
    return float(np.mean((np.sign(fc[i] - fc[j]) > 0) & (np.sign(fqv[i] - fqv[j]) <= 0)))

Deltas = np.array([0.01, 0.02, 0.05, 0.1, 0.2, 0.4])

# (a) scaling of AUC loss and migration in Delta
auc0 = roc_auc_score(y, f); al, mg = [], []
for D in Deltas:
    fqv = fq(D); al.append(max(auc0 - roc_auc_score(y, fqv), 1e-6)); mg.append(migration(f, fqv, 10))
al, mg = np.array(al), np.array(mg)
slope_auc = np.polyfit(np.log(Deltas[2:]), np.log(al[2:]), 1)[0]
slope_mig = np.polyfit(np.log(Deltas), np.log(mg), 1)[0]

# (b) sensitivity to P (typical scale and separation ratio at Delta=0.1)
Ps = [5, 10, 20, 40]; stdP, ratioP = [], []
for Pv in Ps:
    Cm, fc, yy, bb = gen(Pv, seed=100 + Pv)
    fqc = bb + (0.1 * np.round(Cm / 0.1)).sum(1)
    e = fqc - fc; stdP.append(e.std())
    R = reversal(fc, fqc, yy); M = migration(fc, fqc, 10); ratioP.append(M / max(R, 1e-6))
stdP = np.array(stdP)

# (c) banding scheme
schemes = ["freq", "width", "modes"]; Mc = {s: [] for s in schemes}
for D in Deltas:
    fqv = fq(D)
    for s in schemes: Mc[s].append(migration(f, fqv, 10, s))

# (d) genericity of the sqrt(P) typical-scale law across random populations
many = []
for s in range(30):
    Cm, fc, yy, bb = gen(20, seed=500 + s)
    many.append([(bb + (D * np.round(Cm / D)).sum(1) - (bb + Cm.sum(1))).std() for D in Deltas])
many = np.array(many)

pd.DataFrame({"Delta": Deltas, "auc_loss": al, "migration_K10": mg}).to_csv("../results/tables/06_scaling.csv", index=False)
pd.DataFrame({"P": Ps, "std_e": stdP, "std_over_sqrtP": stdP / np.sqrt(Ps), "sep_ratio": ratioP}).to_csv("../results/tables/06_Psensitivity.csv", index=False)
print("scaling slopes (log-log): AUC loss =", round(slope_auc, 2), " migration =", round(slope_mig, 2))
print("P sweep:", Ps)
print("  std(e) at Delta=0.1 :", np.round(stdP, 3))
print("  std(e)/sqrt(P)      :", np.round(stdP / np.sqrt(Ps), 4), "(should be ~constant)")
print("  separation ratio M/R:", np.round(ratioP, 1), "(Delta and sqrt(P) cancel; residual trend is the geometric factor rho/phi_m)")

fig, ax = plt.subplots(2, 2, figsize=(9, 7))
ax[0, 0].plot(Deltas, al, ls="-", color="0.0", marker="o", label=f"AUC loss (slope {slope_auc:.1f})")
ax[0, 0].plot(Deltas, mg, ls="--", color="0.45", marker="s", label=f"migration K=10 (slope {slope_mig:.1f})")
ax[0, 0].set_xscale("log"); ax[0, 0].set_yscale("log")
ax[0, 0].set_xlabel(r"lattice spacing $\Delta$"); ax[0, 0].set_ylabel("loss"); ax[0, 0].legend(fontsize=7)

ax[0, 1].plot(Ps, stdP, ls="-", color="0.0", marker="o", label=r"observed $\mathrm{std}(e)$")
ax[0, 1].plot(Ps, 0.1 * np.sqrt(np.array(Ps) / 12), ls=":", color="0.5", marker="^", label=r"$\Delta\sqrt{P/12}$")
ax[0, 1].set_xlabel("number of features P"); ax[0, 1].set_ylabel(r"$\mathrm{std}(e)$ at $\Delta=0.1$"); ax[0, 1].legend(fontsize=7)

for s, ls, col, mk in zip(schemes, LS, GREYS, MK):
    ax[1, 0].plot(Deltas, Mc[s], ls=ls, color=col, marker=mk, label=s)
ax[1, 0].set_xlabel(r"lattice spacing $\Delta$"); ax[1, 0].set_ylabel("migration rate (K=10)"); ax[1, 0].legend(fontsize=7)

for row in many:
    ax[1, 1].plot(Deltas, row, color="0.78", lw=0.6)
ax[1, 1].plot(Deltas, Deltas * np.sqrt(20 / 12), ls="-", color="0.0", marker="^", label=r"typical $\Delta\sqrt{P/12}$")
ax[1, 1].plot(Deltas, 20 * Deltas / 2, ls="--", color="0.3", label=r"worst $P\Delta/2$")
ax[1, 1].set_xscale("log"); ax[1, 1].set_yscale("log")
ax[1, 1].set_xlabel(r"lattice spacing $\Delta$"); ax[1, 1].set_ylabel(r"$\mathrm{std}(e)$, 30 random populations"); ax[1, 1].legend(fontsize=7)
print("genericity: std(e)/typical across 30 populations at Delta=0.1 : mean",
      round(float((many[:, 3] / (Deltas[3] * np.sqrt(20 / 12))).mean()), 3),
      "sd", round(float((many[:, 3] / (Deltas[3] * np.sqrt(20 / 12))).std()), 3))
fig.tight_layout(); _save(fig, "06_robustness"); plt.close(fig)

scaling slopes (log-log): AUC loss = 2.33  migration = 1.03
P sweep: [5, 10, 20, 40]
  std(e) at Delta=0.1 : [0.056 0.092 0.127 0.208]
  std(e)/sqrt(P)      : [0.0251 0.0292 0.0285 0.0328] (should be ~constant)
  separation ratio M/R: [10.5 14.7 17.5 29.8] (Delta and sqrt(P) cancel; residual trend is the geometric factor rho/phi_m)
genericity: std(e)/typical across 30 populations at Delta=0.1 : mean 1.042 sd 0.086
